# Verne, small, step 06: DAgger (a data step), on T1-wide

The model is step 04's, unchanged. Step 05: `05_square.ipynb`. How DAgger's code works: `06_dagger_explained.ipynb`.

Step 05's best demos (square-up + full shake) leave a policy that fails in states no demo shows: a crooked hold it won't lift, a finger on top of the cube, a missed grasp it never retries (no expert demo ever retries). DAgger labels the policy's own states:

```-
round k:  the round k-1 policy drives on fresh seeds (2000+, never the eval seeds 1000-1049)
          StateExpert labels every step it reaches (what to do from here)
          retrain from scratch on base demos + the labelled rounds' episodes
```

- **Labeller** (`StateExpert`, `expert.py`): reads its phase from the scene each step (cube held? in the bin? closed on nothing?) with a little memory (a settled grip stays held, release is final, phase changes need two agreeing steps), and retries: a grip closed on nothing or a cube that slips out → open, rise clear above the cube, re-aim. The scripted demo expert (`ExpertPolicy`) only moves forward, so after a miss it would carry an empty gripper and call that right.
- **Collection** (`dagger.py`, `run dagger`): the usual recorder, the other way round: it records the labeller's action and executes the policy's. Every episode kept, failures included.
- **Training data** (`run.py` `load_task`): `base+round[:fail]+...`; `:fail` keeps only the episodes the driving policy failed. Validation stays the base's 10 held-out demos.

Seed 0 throughout, local RTX 4060, 50 rollout seeds at n_exec 8 and 16.

In [ ]:
import json

from dream_robot.policies.verne.data import REPO
from dream_robot.policies.verne.run import RUNS

BASE = 't1-wide-square-noise'
RUNS_06 = {
    'base: square + shake (step 05)': RUNS / '05_square' / BASE / 'seed0',
    '+ r1, all 100 episodes': RUNS / '06_dagger' / f'{BASE}+dagger_r1' / 'seed0',
    '+ r1, failed only': RUNS / '06_dagger' / f'{BASE}+dagger_r1:fail' / 'seed0',
    '+ r1 + r2, failed only': RUNS
    / '06_dagger'
    / f'{BASE}+dagger_r1:fail+dagger_r2:fail'
    / 'seed0',
}

### step 06.1 — the labeller

Checked before any collection (`experiments/verne/scripts/state_expert_check.py`):
- alone, on T1-wide spawns 0-29: **30/30 placed**, phase sequences as clean as the scripted expert's (7-10 phase changes an episode);
- taking over step 05's clean-squaring policy (`t1-wide-square`, seed 0) in its 22 failed eval episodes, 40 steps after the policy's first close: **21/21 placed** (the 22nd had ended under the policy), crooked holds carried on, missed grasps retried.

Two design fixes on the way: without memory it flip-flopped (lift ↔ carry ↔ lower, grasp ↔ lift; 16/30 alone); and its first retries re-closed in place, grinding on the cube, until reopen was made to rise clear above the cube before re-aiming.

### step 06.2 — the collected rounds

In [ ]:
for name, policy in (('dagger_r1', 'base'), ('dagger_r2', '+ r1, failed only')):
    s = json.loads((REPO / 'data/robosuite' / name / 'recording_summary.json').read_text())
    failed = sum(not e['success'] for e in s['episodes'])
    print(
        f"{name}: driven by '{policy}', seeds {s['episodes'][0]['seed']}-{s['episodes'][-1]['seed']}: "
        f'{s["kept"]} episodes, {s["total_frames"]} frames, the policy placed {s["expert_success_rate"]:.0%}, '
        f'failed {failed} {s["failure_histogram"]}'
    )

dagger_r1: driven by 'base', seeds 2000-2099: 100 episodes, 33062 frames, the policy placed 79%, failed 21 {'dropped': 7, 'no_grasp': 14}
dagger_r2: driven by '+ r1, failed only', seeds 2100-2299: 200 episodes, 70646 frames, the policy placed 76%, failed 47 {'dropped': 8, 'no_grasp': 35, 'wrong_target': 4}


### step 06.3 — seed 0 after each round

In [ ]:
def read_run(path):
    # a run folder -> n_exec -> (successes, failure counts), probe fit, final val loss
    out = {}
    for n in (8, 16):
        res = json.loads((path / f'sweep_{n}.json').read_text()).values()
        fails = {}
        for ok, mode, _ in res:
            if not ok:
                fails[mode] = fails.get(mode, 0) + 1
        out[n] = (sum(r[0] for r in res), fails)
    log = json.loads((path / 'log.json').read_text())
    return out, json.loads((path / 'probe.json').read_text())['fit'], log[-1][2]

In [ ]:
print(
    f'{"":32s} {"n_exec 8":>9s} {"n_exec 16":>10s} {"miss":>7s} {"y slope":>8s}  failures (n_exec 8 | 16)'
)
for name, path in RUNS_06.items():
    r, fit, val = read_run(path)
    print(
        f'{name:32s} {r[8][0]:6d}/50 {r[16][0]:7d}/50 {fit["miss_cm"]:5.1f} cm {fit["slope_y"]:+8.2f}  '
        f'{r[8][1]} | {r[16][1]}'
    )

                                  n_exec 8  n_exec 16    miss  y slope  failures (n_exec 8 | 16)
base: square + shake (step 05)       42/50      44/50   1.0 cm    +0.86  {'no_grasp': 6, 'wrong_target': 1, 'dropped': 1} | {'no_grasp': 6}
+ r1, all 100 episodes               45/50      44/50   1.0 cm    +0.94  {'no_grasp': 3, 'dropped': 1, 'wrong_target': 1} | {'dropped': 2, 'wrong_target': 2, 'no_grasp': 2}
+ r1, failed only                    45/50      47/50   1.0 cm    +0.90  {'dropped': 2, 'no_grasp': 3} | {'no_grasp': 3}
+ r1 + r2, failed only               48/50      47/50   0.9 cm    +0.97  {'wrong_target': 2} | {'dropped': 2, 'no_grasp': 1}


### step 06.4 — what's left (round 2's model)

- All 5 failures (2 at n_exec 8, 3 at 16) ran to the 600-step limit (20 s; a success takes ~260).
- Replayed (`scratch/film_eps.py`, same GPU, same per-episode seed): **4 of 5 placed**, within 600 steps; with a 40 s limit too (`rollouts.make_env(horizon_seconds=...)`), the same 4 place in 280-549 steps. They are knife-edge episodes that flip with run-to-run GPU differences, not a time limit.
- Ep 1001 (n_exec 16, cube −29°) fails every time, even at 40 s: the cube slips out early in the lift, the policy goes back and grips it, but crooked (the fall turned it), and holds it there without lifting for 800 steps: step 05's crooked-hold stall, now after a retry.
- Earlier rounds' films: the round-1 failures-only model's misses had a finger come down on top of the cube; round 2's failures (47 episodes, many of those) removed them.

### wrap-up — DAgger works, and failures are the data that matters

**Buys:** seed 0 goes 42 / 44 (step 05's best demos) → 45 / 47 (round 1, failures only) → **48 / 47** (round 2), missed grasps at n_exec 8 from 6 to 0, the sideways aim from y slope 0.86 to 0.97, miss 0.9 cm. Keeping only the failed episodes matches or beats keeping all (45 / 47 against 45 / 44 from round 1's data, with a third fewer frames): the 79 successes added nothing measurable. The labeller is what makes it possible: the scripted expert can't label off-path states, and no demo ever showed a retry.

**Where it fails:** one stubborn case (a re-grasp after a drop, gripped crooked, never lifted); and the rest is noise: 2-3 episodes per sweep now flip between runs on the same GPU, so single sweeps can't separate models any finer.

**Not answered:** seed 0 only; whether more rounds keep helping or saturate; the policy places only 76% of collection episodes against 90% on the eval seeds (seed difficulty or a collection mismatch, unchecked); T1 not run.